<a href="https://colab.research.google.com/github/RMoulla/GIA_Nanterre/blob/main/Fusion_Gemma4_MASSIVE_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fusionner Gemma 4 et l’adaptateur MASSIVE français

Ce notebook intègre les modifications apprises par l’adaptateur LoRA dans les poids de Gemma 4 E2B. Il produit un **modèle Transformers complet**, qui se chargera sans adaptateur séparé. La dernière section convertit ensuite ce modèle en **GGUF Q4_K_M** et prépare son import dans **Ollama**. Aucun entraînement n’est relancé.

Avant de commencer :

1. Dans **Exécution → Modifier le type d’exécution**, choisir un **GPU T4**.
2. Utiliser une session libre. Si le modèle d’entraînement est encore en mémoire, **redémarrer la session** pour libérer le GPU, sans supprimer les fichiers.
3. Garder l’archive `gemma4_massive_fr_1000_resultats.zip` disponible : elle sera demandée si les résultats ne sont plus dans `/content`.

Exécuter les cellules dans l’ordre. La base est téléchargée à la révision exacte de l’expérience, en **FP16**. Prévoir environ **25 Go de disque libre** au départ ; le modèle final occupera environ **10 Go**. Le stockage de Colab est temporaire : conserver la sortie avant de supprimer la session.


## 1. Installer les bibliothèques

Nous gardons les versions utilisées pour cette expérience. `transformers` charge Gemma ; `peft` lit l’adaptateur et réalise la fusion. La base sera rechargée en FP16, sans quantification 4 bits pendant la fusion.


In [ ]:
%pip -q install "transformers==5.10.1" "peft==0.19.0" "accelerate==1.14.0" sentencepiece


## 2. Choisir les dossiers

`results_dir` désigne les résultats de l’entraînement. `output_dir` recevra le modèle complet ; il doit être vide ou ne pas encore exister. Les deux constantes de modèle identifient précisément la base de notre adaptateur.


In [ ]:
from pathlib import Path
import json
import shutil

results_dir = Path('/content/gemma4_massive_fr_1000').resolve()
adapter_dir = results_dir / 'adapter'
output_dir = Path('/content/gemma4_massive_fr_fusionne').resolve()

MODEL_ID = 'google/gemma-4-E2B-it'
REVISION = '3e22461f65e89153144f8adb70e3b8c2cc9845a7'

if output_dir == adapter_dir or adapter_dir in output_dir.parents:
    raise ValueError('La sortie doit être placée en dehors du dossier adapter.')
if output_dir.exists() and (not output_dir.is_dir() or any(output_dir.iterdir())):
    raise FileExistsError(f'Choisir un dossier de sortie vide : {output_dir}')


## 3. Retrouver les résultats de l’entraînement

Nous avons besoin des poids LoRA, du tokenizer, de la consigne et de la configuration de l’expérience. Ce contrôle évite de fusionner les poids tout en perdant le format des requêtes utilisé à l’entraînement.


In [ ]:
required_files = [
    'adapter/adapter_config.json',
    'adapter/adapter_model.safetensors',
    'adapter/tokenizer.json',
    'adapter/tokenizer_config.json',
    'adapter/chat_template.jinja',
    'system_prompt.txt',
    'run_config.json',
]
missing_files = [name for name in required_files if not (results_dir / name).is_file()]
print('Fichiers manquants :', missing_files or 'aucun')


### Importer l’archive si nécessaire

Si tous les fichiers sont déjà présents, cette cellule ne demande rien. Sinon, sélectionner uniquement l’archive ZIP des résultats de notre entraînement.


In [ ]:
archive_path = None
if missing_files:
    from google.colab import files

    print('Sélectionner gemma4_massive_fr_1000_resultats.zip.')
    uploaded = files.upload()
    names = list(uploaded)
    del uploaded  # Libérer la copie de l’archive conservée en mémoire.
    if len(names) != 1 or not names[0].lower().endswith('.zip'):
        raise ValueError('Importer exactement une archive ZIP des résultats.')
    archive_path = Path(names[0])
else:
    print('Les résultats sont déjà disponibles dans', results_dir)


### Extraire les fichiers utiles

La cellule suivante vérifie le contenu du ZIP avant de recopier l’adaptateur et ses métadonnées dans `results_dir`. Les prédictions et les journaux d’entraînement ne sont pas nécessaires à la fusion.


In [ ]:
from zipfile import ZipFile

if archive_path is not None:
    with ZipFile(archive_path) as archive:
        absent = [name for name in required_files if name not in archive.namelist()]
        if absent:
            raise FileNotFoundError(f'Archive incomplète ; fichiers absents : {absent}')
        members = [item for item in archive.infolist() if not item.is_dir() and (
            item.filename.startswith('adapter/')
            or item.filename in ('system_prompt.txt', 'run_config.json')
        )]
        for item in members:
            target = (results_dir / item.filename).resolve()
            if not target.is_relative_to(results_dir):
                raise ValueError(f'Chemin invalide dans le ZIP : {item.filename}')
        for item in members:
            target = results_dir / item.filename
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.open(item) as source, target.open('wb') as destination:
                shutil.copyfileobj(source, destination)

missing_files = [name for name in required_files if not (results_dir / name).is_file()]
if missing_files:
    raise FileNotFoundError(f'Résultats incomplets : {missing_files}')
print('Adaptateur disponible :', adapter_dir)


## 4. Vérifier la base et la révision

Un adaptateur LoRA correspond à une base précise. Les informations de l’adaptateur **et** celles de l’entraînement doivent indiquer le même modèle et la même révision. En cas d’erreur ici, reprendre l’archive originale de l’expérience.


In [ ]:
adapter_config = json.loads((adapter_dir / 'adapter_config.json').read_text(encoding='utf-8'))
run_config = json.loads((results_dir / 'run_config.json').read_text(encoding='utf-8'))
expected_base = (MODEL_ID, REVISION)
adapter_base = (adapter_config.get('base_model_name_or_path'), adapter_config.get('revision'))
training_base = (run_config.get('model_id'), run_config.get('model_revision'))

if adapter_base != expected_base:
    raise ValueError(f'Base de l’adaptateur incorrecte : {adapter_base}. Attendu : {expected_base}')
if training_base != expected_base:
    raise ValueError(f'Base de l’entraînement incorrecte : {training_base}. Attendu : {expected_base}')
if adapter_config.get('peft_type') != 'LORA':
    raise ValueError('Cet adaptateur doit être de type LORA.')

system_prompt = (results_dir / 'system_prompt.txt').read_text(encoding='utf-8')
if not system_prompt.strip():
    raise ValueError('La consigne system_prompt.txt est vide ; reprendre l’archive originale.')
print('Base et révision confirmées :', MODEL_ID, REVISION)


## 5. Vérifier que le T4 est libre

Le chargement FP16 utilise plus de mémoire que le modèle QLoRA 4 bits. Nous demandons au moins **12 Gio de VRAM libre** avant de charger la base. Si le contrôle échoue, redémarrer la session Colab, puis reprendre ce notebook depuis le début.


In [ ]:
import gc
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Activer le GPU T4 dans le type d’exécution Colab.')
gpu_name = torch.cuda.get_device_name(0)
if 'T4' not in gpu_name:
    raise RuntimeError(f'Ce notebook est prévu pour un T4 ; GPU actuel : {gpu_name}')
gc.collect()
torch.cuda.empty_cache()
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print(f'{gpu_name} : {free_bytes / 2**30:.2f} Gio libres sur {total_bytes / 2**30:.2f} Gio.')
if free_bytes < 12 * 2**30:
    raise RuntimeError('Mémoire insuffisante : redémarrer la session pour libérer le modèle précédent.')
print(f'Disque libre dans /content : {shutil.disk_usage("/content").free / 1e9:.1f} Go.')


## 6. Charger la base en FP16

La classe `Gemma4ForConditionalGeneration` conserve la même architecture que pendant l’entraînement. Les poids audio et vision restent présents ; l’adaptateur de notre expérience porte sur les couches du langage.

Le téléchargement peut prendre quelques minutes. Si Hugging Face demande une authentification, appeler `huggingface_hub.login()` dans une cellule, puis relancer ce chargement. Ne pas écrire de token dans le notebook.


In [ ]:
from transformers import AutoTokenizer, Gemma4ForConditionalGeneration

# Le tokenizer vient de l’adaptateur : on conserve exactement son template.
tokenizer = AutoTokenizer.from_pretrained(str(adapter_dir), local_files_only=True)
print('Chargement de la base en FP16…')
base = Gemma4ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=REVISION,
    dtype=torch.float16,
    device_map={'': 0},
    attn_implementation='sdpa',
)
base.eval()
base.requires_grad_(False)
print('Base chargée :', type(base).__name__)


## 7. Ajouter l’adaptateur appris

À ce stade, la base et les modifications LoRA sont encore séparées. `is_trainable=False` indique que nous n’allons pas reprendre l’entraînement.


In [ ]:
from peft import PeftModel

adapted = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)
adapted.eval()
adapted.requires_grad_(False)
lora_names = [name for name, _ in adapted.named_parameters() if 'lora_' in name]
if not lora_names:
    raise RuntimeError('Aucun poids LoRA chargé : vérifier l’archive avant de continuer.')
print(f'Adaptateur chargé : {len(lora_names)} tenseurs LoRA.')


## 8. Fusionner les poids

`merge_and_unload` intègre les modifications LoRA dans les poids de la base, puis retire les couches d’adaptation. L’option `safe_merge=True` contrôle la présence de valeurs non finies pendant la fusion.


In [ ]:
with torch.no_grad():
    merged = adapted.merge_and_unload(safe_merge=True)

if any('lora_' in name for name, _ in merged.named_parameters()):
    raise RuntimeError('Des poids LoRA subsistent : ne pas sauvegarder ce modèle.')
merged.eval()
merged.config.use_cache = True
merged.config.get_text_config().use_cache = True
print('Fusion terminée en mémoire : les couches LoRA ont été retirées.')


## 9. Sauvegarder le modèle complet

La sauvegarde vise des fichiers de 2 Go. Un tenseur individuel plus grand, comme la table d’embeddings de Gemma, reste dans un fichier plus volumineux. Nous enregistrons aussi le tokenizer, la consigne MASSIVE et la configuration d’entraînement.

Cette cellule refuse toute sortie non vide. Si une sauvegarde a été interrompue, choisir un nouveau `output_dir` vide avant de réessayer.


In [ ]:
if output_dir.exists() and (not output_dir.is_dir() or any(output_dir.iterdir())):
    raise FileExistsError(f'La sortie existe déjà : choisir un nouveau dossier vide. {output_dir}')
output_dir.mkdir(parents=True, exist_ok=True)

merged.save_pretrained(str(output_dir), max_shard_size='2GB')
tokenizer.save_pretrained(str(output_dir))
for filename in ('system_prompt.txt', 'run_config.json'):
    shutil.copy2(results_dir / filename, output_dir / filename)

print('Modèle complet sauvegardé dans', output_dir)


### Garder une trace de la fusion

Le fichier `fusion_metadata.json` relie le résultat à la révision de Gemma et à l’adaptateur utilisé. Son empreinte SHA-256 identifie les poids LoRA sans les recopier dans le modèle final.


In [ ]:
import hashlib
from datetime import datetime, timezone
from importlib.metadata import version

with (adapter_dir / 'adapter_model.safetensors').open('rb') as stream:
    adapter_sha256 = hashlib.file_digest(stream, 'sha256').hexdigest()
metadata = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'base_model': MODEL_ID,
    'base_revision': REVISION,
    'adapter_sha256': adapter_sha256,
    'base_load_dtype': 'float16',
    'base_quantized_during_merge': False,
    'safe_merge': True,
    'gpu': gpu_name,
    'packages': {name: version(name) for name in ('torch', 'transformers', 'peft', 'accelerate')},
    'quality_evaluation': 'Pas d’évaluation du corpus après fusion ; les scores QLoRA NF4 ne sont pas transférés automatiquement.',
}
(output_dir / 'fusion_metadata.json').write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2) + '\n', encoding='utf-8'
)
print('Traçabilité enregistrée.')


## 10. Contrôler les fichiers enregistrés

Le contrôle vérifie les fichiers essentiels et la présence de chaque morceau de poids annoncé dans l’index. Il complète les contrôles de fusion ; il ne mesure pas la qualité des prédictions.


In [ ]:
expected_files = [
    'config.json', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja',
    'system_prompt.txt', 'run_config.json', 'fusion_metadata.json',
]
index_path = output_dir / 'model.safetensors.index.json'
if index_path.is_file():
    weight_index = json.loads(index_path.read_text(encoding='utf-8'))
    expected_files += sorted(set(weight_index['weight_map'].values()))
else:
    expected_files.append('model.safetensors')
missing_output = [name for name in expected_files if not (output_dir / name).is_file()]
if missing_output:
    raise FileNotFoundError(f'Sauvegarde incomplète : {missing_output}')
weight_size = sum(path.stat().st_size for path in output_dir.glob('*.safetensors'))
print(f'Fichiers présents. Taille des poids : {weight_size / 1e9:.2f} Go.')
print('Dossier à conserver :', output_dir)


## 11. Essayer une phrase — facultatif

Nous réutilisons le modèle fusionné déjà en mémoire. La génération se fait **une phrase à la fois, sans padding**, comme dans l’évaluation corrigée de l’expérience. Modifier `phrase` pour faire un autre essai.


In [ ]:
phrase = 'Allume la lumière du salon.'
messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': phrase},
]
prompt = tokenizer.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True, enable_thinking=False,
)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to('cuda:0')
with torch.inference_mode():
    generated = merged.generate(
        **inputs, max_new_tokens=20, do_sample=False,
        eos_token_id=tokenizer.convert_tokens_to_ids('<turn|>'),
        pad_token_id=tokenizer.pad_token_id, use_cache=True,
    )
answer = tokenizer.decode(generated[0, inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
print('Phrase :', phrase)
print('Intention prédite :', answer)


Le résultat dans `/content/gemma4_massive_fr_fusionne` est un checkpoint Transformers complet : les modifications apprises font maintenant partie des poids. Conserver **tout le dossier**, y compris les fichiers du tokenizer et `system_prompt.txt`.

Une phrase permet de vérifier le fonctionnement, mais il faut refaire l’évaluation sur le corpus pour connaître la qualité du modèle fusionné en FP16. Les scores obtenus avec la base quantifiée NF4 et l’adaptateur séparé ne décrivent pas automatiquement ce nouveau checkpoint.

Références : [fusion d’un adaptateur avec PEFT](https://huggingface.co/docs/peft/v0.19.0/en/package_reference/peft_model#peft.PeftModel.merge_and_unload), [configuration de la base à la révision utilisée](https://huggingface.co/google/gemma-4-E2B-it/blob/3e22461f65e89153144f8adb70e3b8c2cc9845a7/config.json).


## Conversion au format GGUF utilisable avec Ollama

Cette section poursuit le travail à partir du **modèle fusionné et sauvegardé**. Elle produit un fichier GGUF quantifié et un `Modelfile`, à télécharger puis importer dans Ollama sur votre ordinateur.

Le parcours est : **modèle fusionné → GGUF FP16 → GGUF Q4_K_M → import Ollama**. La conversion conserve la partie texte nécessaire à MASSIVE ; nous n’exportons pas les composants image et audio. Le T4 n’est plus nécessaire pour ces opérations, qui utilisent le CPU.

Prévoir environ **30 Go libres supplémentaires** pour les fichiers intermédiaires et l’archive. La quantification peut modifier les prédictions : les scores du modèle QLoRA d’origine ne sont pas automatiquement ceux du GGUF.

### 1. Préparer les fichiers de sortie

Nous utilisons le dossier fusionné de la section précédente. Le GGUF FP16 sera un intermédiaire ; seul le GGUF Q4_K_M sera inclus dans le dossier à importer dans Ollama. Nous gardons les fichiers Safetensors d’origine.

In [ ]:
from pathlib import Path
import gc
import os
import subprocess
import sys

assert (output_dir / 'fusion_metadata.json').is_file(), 'Exécuter d’abord la fusion et la sauvegarde.'
export_dir = Path('/content/gemma4_massive_fr_ollama')
export_dir.mkdir(exist_ok=True)
gguf_fp16 = Path('/content/gemma4_massive_fr-F16.gguf')
gguf_q4 = export_dir / 'gemma4-massive-fr-Q4_K_M.gguf'

for filename in ('base', 'adapted', 'merged', 'inputs', 'generated'):
    globals().pop(filename, None)  # Le modèle est déjà sauvegardé sur disque.
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Dossier destiné à Ollama :', export_dir)

### 2. Récupérer llama.cpp

**llama.cpp** fournit le convertisseur GGUF et l’outil de quantification. Nous fixons une révision précise qui prend en charge Gemma 4, pour conserver le même code de conversion lors d’une nouvelle exécution.

In [ ]:
LLAMA_REVISION = 'bd4f514db14d87fded667787a7a963bfbaa98e89'
llama_dir = Path('/content/llama.cpp-gemma4-export')
if not llama_dir.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1',
        'https://github.com/ggml-org/llama.cpp.git', str(llama_dir),
    ], check=True)
subprocess.run(['git', '-C', str(llama_dir), 'fetch', '--depth', '1', 'origin', LLAMA_REVISION], check=True)
subprocess.run(['git', '-C', str(llama_dir), 'checkout', '--detach', LLAMA_REVISION], check=True)
revision = subprocess.check_output(['git', '-C', str(llama_dir), 'rev-parse', 'HEAD'], text=True).strip()
assert revision == LLAMA_REVISION
print('Révision llama.cpp :', revision)

### 3. Installer le lecteur GGUF et compiler le quantificateur

Nous installons le paquet `gguf` du même dépôt, puis compilons seulement `llama-quantize`, sur CPU. Les commandes s’arrêtent en cas d’erreur grâce à `check=True`.

Nous conservons Transformers 5.10.1 utilisé plus haut : le fichier général de dépendances de llama.cpp imposerait une autre version. Installer `gguf-py` n’impose pas de remplacer Transformers ni PyTorch.

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    str(llama_dir / 'gguf-py'), 'cmake',
], check=True)
build_dir = llama_dir / 'build'
subprocess.run([
    'cmake', '-S', str(llama_dir), '-B', str(build_dir),
    '-DCMAKE_BUILD_TYPE=Release', '-DGGML_CUDA=OFF', '-DLLAMA_BUILD_TESTS=OFF',
    '-DLLAMA_BUILD_SERVER=OFF', '-DLLAMA_BUILD_APP=OFF', '-DLLAMA_OPENSSL=OFF',
], check=True)
subprocess.run([
    'cmake', '--build', str(build_dir), '--target', 'llama-quantize', '-j', '2',
], check=True)
quantizer = build_dir / 'bin' / 'llama-quantize'
assert quantizer.is_file()

### 4. Convertir le modèle fusionné en GGUF FP16

Le convertisseur lit **notre dossier fusionné**, et non le modèle de base téléchargé. Le GGUF contient donc les modifications apprises sur MASSIVE.

`--outtype f16` conserve la précision FP16 à cette étape. `--use-temp-file` permet au convertisseur d’utiliser un fichier temporaire pour limiter la RAM. Ne pas ajouter `--mmproj` : ce paramètre exporterait la partie multimodale, inutile pour notre classifieur de texte.

In [ ]:
cpu_environment = {**os.environ, 'CUDA_VISIBLE_DEVICES': ''}
subprocess.run([
    sys.executable, str(llama_dir / 'convert_hf_to_gguf.py'), str(output_dir),
    '--outfile', str(gguf_fp16), '--outtype', 'f16', '--use-temp-file',
], check=True, env=cpu_environment)
assert gguf_fp16.is_file() and gguf_fp16.stat().st_size > 0
print(f'GGUF FP16 : {gguf_fp16.stat().st_size / 1e9:.2f} Go')

### 5. Réduire la taille avec Q4_K_M

Q4_K_M utilise une quantification principalement sur 4 bits, avec une précision plus élevée pour certains poids. Elle est différente de la quantification NF4 utilisée pendant l’entraînement. Nous quantifions le GGUF FP16 en une seule étape.

Ollama importera le GGUF déjà quantifié ; il n’effectue pas cette quantification lors de l’import.

In [ ]:
subprocess.run([
    str(quantizer), str(gguf_fp16), str(gguf_q4), 'Q4_K_M',
], check=True, env=cpu_environment)
assert gguf_q4.is_file() and gguf_q4.stat().st_size > 0
print(f'GGUF destiné à Ollama : {gguf_q4.stat().st_size / 1e9:.2f} Go')

### 6. Vérifier le format du fichier

Le lecteur GGUF contrôle que le fichier peut être lu et contient des tenseurs de l’architecture Gemma 4 avec le type Q4_K_M. Ce contrôle de format ne remplace pas un essai de génération dans Ollama.

In [ ]:
from gguf import GGUFReader, LlamaFileType

reader = GGUFReader(str(gguf_q4))
architecture = reader.get_field('general.architecture')
file_type = reader.get_field('general.file_type')
assert architecture is not None and architecture.contents() == 'gemma4'
assert file_type is not None and file_type.contents() == int(LlamaFileType.MOSTLY_Q4_K_M)
assert len(reader.tensors) > 0
print('GGUF lisible :', len(reader.tensors), 'tenseurs ; Gemma 4 ; Q4_K_M.')
del reader
gc.collect()

### 7. Créer le Modelfile pour Ollama

Le `Modelfile` est un **fichier texte propre à Ollama**, sans extension `.txt`. `FROM` pointe vers le GGUF placé dans le même dossier. `SYSTEM` reprend exactement notre consigne et les 60 intentions autorisées.

Nous choisissons une température nulle et une réponse courte. Ollama utilise son format de conversation Gemma 4 ; la commande d’essai ci-dessous désactive explicitement le raisonnement avec `--think=false`, comme à l’entraînement.

In [ ]:
system_prompt = (output_dir / 'system_prompt.txt').read_text(encoding='utf-8')
assert system_prompt.strip() and '"""' not in system_prompt
modelfile = (
    f'FROM ./{gguf_q4.name}\n'
    f'SYSTEM """{system_prompt}"""\n'
    'PARAMETER temperature 0\n'
    'PARAMETER num_ctx 512\n'
    'PARAMETER num_predict 20\n'
    'PARAMETER repeat_penalty 1.0\n'
    'PARAMETER stop "<turn|>"\n'
    'PARAMETER stop "<eos>"\n'
)
(export_dir / 'Modelfile').write_text(modelfile, encoding='utf-8')
shutil.copy2(output_dir / 'system_prompt.txt', export_dir / 'system_prompt.txt')
shutil.copy2(output_dir / 'fusion_metadata.json', export_dir / 'fusion_metadata.json')
print(modelfile)

### 8. Télécharger le dossier prêt à importer

L’archive contient le **GGUF Q4_K_M**, le **Modelfile**, la consigne et les informations de fusion/conversion. Elle exclut le gros GGUF FP16 intermédiaire. La création du ZIP lit les fichiers progressivement, sans charger les poids en mémoire.

Le téléchargement peut prendre du temps : le fichier complet pèse plusieurs Go. Conserver l’archive avant de fermer la session Colab.

In [ ]:
import zipfile
from google.colab import files

conversion_metadata = {
    'llama_cpp_revision': LLAMA_REVISION,
    'source_checkpoint': str(output_dir),
    'quantization': 'Q4_K_M',
    'export': 'text-only',
    'ollama_inference_tested': False,
}
(export_dir / 'conversion_metadata.json').write_text(
    json.dumps(conversion_metadata, ensure_ascii=False, indent=2), encoding='utf-8',
)
archive_path = Path('/content/gemma4_massive_fr_ollama.zip')
package_files = [gguf_q4.name, 'Modelfile', 'system_prompt.txt', 'fusion_metadata.json', 'conversion_metadata.json']
with zipfile.ZipFile(archive_path, 'w', zipfile.ZIP_STORED, allowZip64=True) as archive:
    for name in package_files:
        archive.write(export_dir / name, arcname=f'gemma4_massive_fr_ollama/{name}')
print('Archive créée :', archive_path)
files.download(str(archive_path))

### 9. Importer et essayer le modèle sur votre ordinateur

Ces commandes sont à exécuter dans le **Terminal de votre ordinateur**, après installation d’une version récente d’Ollama prenant en charge Gemma 4 et ouverture de l’application Ollama. Elles ne sont pas à exécuter dans Colab.

Décompresser `gemma4_massive_fr_ollama.zip`, ouvrir un terminal dans le dossier `gemma4_massive_fr_ollama` contenant le GGUF et le Modelfile, puis lancer :

```bash
ollama create gemma4-e2b-massive-fr -f Modelfile
ollama run gemma4-e2b-massive-fr --think=false "Allume la lumière du salon."
```

Le modèle doit répondre par une étiquette MASSIVE, par exemple `iot_hue_lighton` pour cette phrase. Cet exemple exprime la réponse attendue ; ce n’est pas une prédiction déjà mesurée après conversion.

Pour obtenir une requête indépendante sans historique via l’API locale :

```bash
curl http://localhost:11434/api/generate -d '{
  "model": "gemma4-e2b-massive-fr",
  "prompt": "Allume la lumière du salon.",
  "think": false,
  "stream": false
}'
```

**À vérifier après exécution :** génération d’une étiquette valide, puis mesure sur les mêmes 300 exemples de test pour connaître l’effet de la fusion et de la quantification. La réussite des cellules de conversion ne suffit pas à garantir le score de classification.

Sources : [conversion et quantification dans llama.cpp](https://github.com/ggml-org/llama.cpp/blob/bd4f514db14d87fded667787a7a963bfbaa98e89/tools/quantize/README.md), [import GGUF dans Ollama](https://docs.ollama.com/import), [format Modelfile](https://docs.ollama.com/modelfile), [désactiver le raisonnement dans Ollama](https://docs.ollama.com/capabilities/thinking).